# Assignment-03 - Task 1: Text Classification with Hugging Face Pipelines

| | |
|---|---|
| **Course:** | PROG74040 - Advanced Topics in Artificial Intelligence and Machine Learning |
| **Group Number:** | Group 5 |
| **Task:** | Task 1 - Sentiment Analysis (Text Classification) |

---

| Step | Description |
|------|-------------|
| 1 | Common setup: imports, device, reproducibility |
| 2 | Baseline: off-the-shelf sentiment-analysis pipeline on sample texts |
| 3 | Fine-tuning configuration (hyperparameters) |
| 4 | Load IMDb and build train/eval subsets |
| 5 | Tokenize the text |
| 6 | Fine-tune a custom DistilBERT classifier |
| 7 | Evaluate on the held-out subset |
| 8 | Save the fine-tuned model and tokenizer |
| 9 | Reload with the `pipeline` function and classify texts |


---
## Section 1: Common Setup

Imports, device selection, and fixed seeds so results are repeatable across runs.

In [1]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    pipeline,
)
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

C:\Users\kahan\CS-Conestoga\sem8\adv_aiml\assignments\aiml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


---
## Section 2: Baseline Pipeline (No Training)

Before training anything, we use the `pipeline` function with its default sentiment model.
This is the "basic usage" the task asks for: one line to load a pretrained model plus its
tokenizer, then classify raw strings. It gives us a reference point to compare our own
fine-tuned model against later.

In [2]:
sample_texts = [
    "This movie was fantastic, easily the best I have seen all year.",
    "A complete waste of time. The plot made no sense and the acting was wooden.",
    "It was okay, not great but not terrible either.",
    "I absolutely loved the soundtrack and the visuals were stunning.",
]

# default model for this task is a DistilBERT fine-tuned on SST-2
baseline = pipeline("sentiment-analysis", device=-1)

print('Baseline pipeline predictions:')
for text, out in zip(sample_texts, baseline(sample_texts)):
    print(f"  [{out['label']:<8} {out['score']:.3f}]  {text[:60]}...")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1175.50it/s]

Baseline pipeline predictions:


  [POSITIVE 1.000]  This movie was fantastic, easily the best I have seen all ye...
  [NEGATIVE 1.000]  A complete waste of time. The plot made no sense and the act...
  [POSITIVE 0.991]  It was okay, not great but not terrible either....
  [POSITIVE 1.000]  I absolutely loved the soundtrack and the visuals were stunn...


---
## Section 3: Fine-Tuning Configuration

All tunable values are here so behavior can be changed without touching the rest of the notebook.

| Parameter | Value | Reason |
|-----------|-------|--------|
| `CHECKPOINT` | distilbert-base-uncased | Smaller and faster than BERT, keeps most of the accuracy |
| `MAX_LENGTH` | 128 | Cap on tokens per review; longer reviews are truncated to fit CPU time |
| `TRAIN_SIZE` | 1500 | Subset of the 25,000 train reviews; full set is too slow to fine-tune on CPU |
| `EVAL_SIZE` | 500 | Held-out subset of the test split for scoring |
| `NUM_EPOCHS` | 1 | One pass is enough for a strong signal at this subset size |
| `BATCH_SIZE` | 16 | Fits comfortably in CPU memory |
| `LEARNING_RATE` | 2e-5 | Standard starting point for fine-tuning transformers |
| `WEIGHT_DECAY` | 0.01 | Light regularization to reduce overfitting |

In [3]:
CHECKPOINT    = "distilbert-base-uncased"
MAX_LENGTH    = 128
TRAIN_SIZE    = 1500
EVAL_SIZE     = 500
NUM_EPOCHS    = 1
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
SAVE_DIR      = "saved_models/task1_imdb_distilbert"

print(f'Checkpoint    : {CHECKPOINT}')
print(f'Max length    : {MAX_LENGTH}')
print(f'Train size    : {TRAIN_SIZE}')
print(f'Eval size     : {EVAL_SIZE}')
print(f'Epochs        : {NUM_EPOCHS}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'Learning rate : {LEARNING_RATE}')

Checkpoint    : distilbert-base-uncased
Max length    : 128
Train size    : 1500
Eval size     : 500
Epochs        : 1
Batch size    : 16
Learning rate : 2e-05


---
## Section 4: Load IMDb and Build Subsets

IMDb ships as 25,000 labeled training reviews and 25,000 labeled test reviews, balanced
50/50 between positive (label 1) and negative (label 0). We shuffle with a fixed seed and
take a subset of each split so the run finishes in a reasonable time on CPU.

In [4]:
imdb = load_dataset("stanfordnlp/imdb")

train_ds = imdb["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
eval_ds  = imdb["test"].shuffle(seed=SEED).select(range(EVAL_SIZE))

# quick sanity check on the label balance of our subset
train_pos = sum(train_ds["label"])
print(f'Train subset : {len(train_ds)}  (positive: {train_pos}, negative: {len(train_ds) - train_pos})')
print(f'Eval subset  : {len(eval_ds)}')
print()
print('Example review:')
print(' label:', train_ds[0]["label"])
print(' text :', train_ds[0]["text"][:200], '...')

Train subset : 1500  (positive: 738, negative: 762)
Eval subset  : 500

Example review:
 label: 1
 text : There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F ...


---
## Section 5: Tokenize

The tokenizer turns raw text into the input IDs the model expects. We truncate to
`MAX_LENGTH` tokens. Padding is left to the data collator so each batch is only padded
to the longest review inside that batch instead of a fixed length, which saves compute.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map(tokenize_batch,  batched=True, remove_columns=["text"])

# pads each batch to its own longest sequence at collation time
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print('Tokenized columns:', train_tok.column_names)


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]


Map:  67%|██████▋   | 1000/1500 [00:02<00:01, 371.47 examples/s]


Map: 100%|██████████| 1500/1500 [00:03<00:00, 393.23 examples/s]


Map: 100%|██████████| 1500/1500 [00:03<00:00, 381.68 examples/s]


Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Map: 100%|██████████| 500/500 [00:01<00:00, 444.39 examples/s]


Map: 100%|██████████| 500/500 [00:01<00:00, 423.09 examples/s]

Tokenized columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


---
## Section 6: Fine-Tune a Custom Classifier

We load DistilBERT with a fresh 2-class classification head on top. The base weights are
pretrained; only the head starts random, so fine-tuning teaches the whole model to map
movie reviews to positive or negative. `id2label` is set now so the saved model reports
readable labels later.

`compute_metrics` reports accuracy and macro F1 on the eval subset. The `Trainer` handles
the training loop, batching, and optimizer for us.

In [6]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="task1_trainer_out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=25,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print()
print(f"Final training loss: {train_result.training_loss:.4f}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]


Loading weights:  87%|████████▋ | 87/100 [00:00<00:00, 868.91it/s]


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 870.19it/s]


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\kahan\CS-Conestoga\sem8\adv_aiml\assignments\aiml_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
25,0.688195
50,0.633188
75,0.546337



Final training loss: 0.5936


---
## Section 7: Evaluate

Accuracy and macro F1 on the 500-review held-out subset. Because the split is balanced,
accuracy alone is a fair summary, and F1 confirms the model is not just guessing one class.

In [7]:
metrics = trainer.evaluate()

print('Held-out evaluation:')
print(f"  Accuracy : {metrics['eval_accuracy']:.4f}")
print(f"  Macro F1 : {metrics['eval_f1']:.4f}")
print(f"  Loss     : {metrics['eval_loss']:.4f}")

Training Loss,Validation Loss,Step,Accuracy,F1
0.546337,0.465293,94,0.804000,0.803194


Held-out evaluation:
  Accuracy : 0.8040
  Macro F1 : 0.8032
  Loss     : 0.4653


---
## Section 8: Save the Fine-Tuned Model

Both the model weights and the tokenizer are written to disk. Saving the tokenizer with the
model is what lets the `pipeline` function reload everything from a single directory.

In [8]:
import os
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f'Saved to ./{SAVE_DIR}/')
for fname in sorted(os.listdir(SAVE_DIR)):
    size_kb = os.path.getsize(os.path.join(SAVE_DIR, fname)) / 1024
    print(f'  {fname}  ({size_kb:.1f} KB)')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]

Saved to ./saved_models/task1_imdb_distilbert/
  config.json  (0.8 KB)
  model.safetensors  (261555.2 KB)
  tokenizer.json  (694.8 KB)
  tokenizer_config.json  (0.4 KB)
  training_args.bin  (5.0 KB)


---
## Section 9: Reload with the Pipeline and Classify

This is the main deliverable to: load our own fine-tuned model back through the
`pipeline` function and run it on new text. Passing the saved directory to both `model` and
`tokenizer` rebuilds the full classifier. We reuse the same sample texts from Section 2 so
the fine-tuned output can be compared directly against the baseline.

In [9]:
clf = pipeline(
    "sentiment-analysis",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=-1,
)

print('Fine-tuned model predictions:')
print('-' * 70)
for text, out in zip(sample_texts, clf(sample_texts)):
    print(f"  [{out['label']:<8} {out['score']:.3f}]  {text}")

print()
new_reviews = [
    "The pacing dragged and I nearly fell asleep halfway through.",
    "An emotional, beautifully shot film that stayed with me for days.",
]
print('Predictions on unseen reviews:')
print('-' * 70)
for text, out in zip(new_reviews, clf(new_reviews)):
    print(f"  [{out['label']:<8} {out['score']:.3f}]  {text}")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2218.84it/s]

Fine-tuned model predictions:
----------------------------------------------------------------------


  [POSITIVE 0.784]  This movie was fantastic, easily the best I have seen all year.
  [NEGATIVE 0.770]  A complete waste of time. The plot made no sense and the acting was wooden.
  [NEGATIVE 0.618]  It was okay, not great but not terrible either.
  [POSITIVE 0.729]  I absolutely loved the soundtrack and the visuals were stunning.

Predictions on unseen reviews:
----------------------------------------------------------------------
  [NEGATIVE 0.641]  The pacing dragged and I nearly fell asleep halfway through.
  [POSITIVE 0.771]  An emotional, beautifully shot film that stayed with me for days.


---
## Summary

- The `pipeline` function classifies raw text in a single call, both for the default
  pretrained model and for our own saved model.
- Fine-tuning DistilBERT on a 1,500-review IMDb subset for one epoch on CPU is enough to
  learn movie-review sentiment.
- Saving the model together with its tokenizer is what makes reloading through `pipeline`
  a one-liner.
- Reported accuracy and macro F1 come from a balanced 1,000-review held-out subset, so they
  reflect real generalization rather than training performance.